In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset

# 1. 读取数据
train_df = pd.read_csv("train.csv")
btest_df = pd.read_csv("Btest.csv")


# 2. 划分训练集 / 验证集（比如 9:1）
train_df, val_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=train_df["label"]
)

# 3. 转成 HuggingFace Dataset（方便和 transformers 对接）
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))
btest_dataset = Dataset.from_pandas(btest_df.reset_index(drop=True))

print(train_dataset[0])

{'id': 1558, 'text': '摸奶节是中国云南双柏县鄂家镇彝族传统文化的庆典就是农历的7月14日、15日与16日这三天，包括外来的游人,如果在街上遇见喜欢的女子，都可以摸一摸女子的胸部。姑娘们表面躲躲闪闪，但决无责怪之意因为这是他们这个地区的百姓延续了1000多年的风俗。小伙子以摸到奶为吉祥，姑娘们以被摸奶为幸运和祝福!', 'label': 1}


In [9]:
from transformers import BertTokenizerFast, BertForSequenceClassification

# 选用哈工大 RoBERTa WWM 扩展版
model_name = "hfl/chinese-roberta-wwm-ext"

# 分词器：RoBERTa 这版本质上还是 BERT 架构，用 BertTokenizerFast 即可
tokenizer = BertTokenizerFast.from_pretrained(model_name)

# 二分类任务，num_labels=2
model = BertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at hfl/chinese-roberta-wwm-ext and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
from datasets import Dataset

# 如果你之前已经有 train_dataset/val_dataset/atest_dataset，就不用重复这一步
# 这里只是提醒一下它们是从 pandas 转过来的：
# train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
# val_dataset   = Dataset.from_pandas(val_df.reset_index(drop=True))
# atest_dataset = Dataset.from_pandas(atest_df.reset_index(drop=True))

max_length = 256  # 注意：必须与训练时一致（你 BERT 用 128 就保持 128）

def preprocess_function(examples):
    result = tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=max_length
    )
    return result

train_encoded = train_dataset.map(preprocess_function, batched=True)
val_encoded   = val_dataset.map(preprocess_function, batched=True)
btest_encoded = btest_dataset.map(preprocess_function, batched=True)


Map:   0%|          | 0/3305 [00:00<?, ? examples/s]

Map:   0%|          | 0/368 [00:00<?, ? examples/s]

Map:   0%|          | 0/1225 [00:00<?, ? examples/s]

In [11]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

def compute_metrics(eval_pred):
    # eval_pred 是一个 (logits, labels) 的元组
    logits, labels = eval_pred

    # preds：按最大 logit 取类别，得到 0 / 1 的预测结果
    preds = np.argmax(logits, axis=-1)

    # 准确率
    acc = accuracy_score(labels, preds)

    # F1 值（默认是二分类的 F1）
    f1 = f1_score(labels, preds)

    # AUC：使用正类（标签=1）的连续得分（这里直接用 logits[:, 1]）
    probs = logits[:, 1]
    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        # 当验证集里只有一类样本时，AUC 无法计算，这里防御一下
        auc = float("nan")

    return {"accuracy": acc, "f1": f1, "auc": auc}


In [12]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./roberta_wwm_256_bs32",   # 输出目录
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="./roberta_wwm_256_bs32_logs",
    logging_steps=50,
    fp16=True      # 如遇到报错/不支持 FP16，直接把这一行删掉即可
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_encoded,
    eval_dataset=val_encoded,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)


C:\Users\fall\AppData\Local\Temp\ipykernel_12644\2772987077.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [13]:
# 训练 RoBERTa-wwm-ext 模型
trainer.train()

# 训练完，在验证集上评估
metrics = trainer.evaluate()
print(metrics)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Step,Training Loss
50,0.526400
100,0.286600
150,0.177900
200,0.134100
250,0.072100
300,0.052100
350,0.046400
400,0.020800


{'eval_loss': 0.19802452623844147, 'eval_accuracy': 0.9456521739130435, 'eval_f1': 0.9425287356321839, 'eval_auc': 0.9828326815973457, 'eval_runtime': 1.4053, 'eval_samples_per_second': 261.863, 'eval_steps_per_second': 4.27, 'epoch': 4.0}


In [14]:
import torch
import numpy as np

predictions = trainer.predict(btest_encoded)

logits = predictions.predictions          # shape: [N, 2]
probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()

prob_fake = probs[:, 1]                   # 正类（label=1=假新闻）概率


# 生成提交文件
submission_b = pd.DataFrame({
    "id": btest_df["id"],
    "prob": prob_fake
})

submission_b.to_csv("roberta_submission_b.csv", index=False, encoding="utf-8")
print("Saved: roberta_submission_b.csv")



Saved: roberta_submission_b.csv
